In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.fljoat32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [137]:
learning_rate = 1e-3
batch_size = 64
epochs = 10

In [138]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

In [139]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [140]:
def training_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode 
    # important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #compute predition and loss
        pred = model(X)
        loss = loss_fn(pred,y)

        #Backpropogation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0 :
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [141]:

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode
    # important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size=len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0,0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X,y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1) == y ).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test errorL \n Accuracy: {(100*correct):>0.1f}%, Avg loss:{test_loss:>8f}\n")

In [142]:
for t in range(epochs):
    print(f"Epoch {t+1}\n---------------------------")
    training_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done")


Epoch 1
---------------------------
loss: 2.301460  [   64/60000]
loss: 2.284612  [ 6464/60000]
loss: 2.261727  [12864/60000]
loss: 2.263425  [19264/60000]
loss: 2.240614  [25664/60000]
loss: 2.211156  [32064/60000]
loss: 2.233960  [38464/60000]
loss: 2.184476  [44864/60000]
loss: 2.184046  [51264/60000]
loss: 2.165959  [57664/60000]
Test errorL 
 Accuracy: 46.4%, Avg loss:2.147415

Epoch 2
---------------------------
loss: 2.155165  [   64/60000]
loss: 2.139073  [ 6464/60000]
loss: 2.078307  [12864/60000]
loss: 2.105726  [19264/60000]
loss: 2.042151  [25664/60000]
loss: 1.991055  [32064/60000]
loss: 2.028620  [38464/60000]
loss: 1.934545  [44864/60000]
loss: 1.939475  [51264/60000]
loss: 1.888135  [57664/60000]
Test errorL 
 Accuracy: 52.4%, Avg loss:1.865590

Epoch 3
---------------------------
loss: 1.897826  [   64/60000]
loss: 1.859546  [ 6464/60000]
loss: 1.740185  [12864/60000]
loss: 1.793002  [19264/60000]
loss: 1.675740  [25664/60000]
loss: 1.640472  [32064/60000]
loss: 1.6724